# 60 — Evaluation harness (reusable, model-frugal)

Runs a small evaluation dataset against the COMPLETED pipeline state and persists every result. This notebook `%run`s `30_retrieval_answers` (definitions only — no demo rerun, no model calls at load) and evaluates the **existing** pipeline state: on a fresh schema, run `40_demo_generation` and `45_robustness_checks` first; on the completed state it can run directly.

**Coverage:** latest-version selection, changed filenames, late arrivals, duplicates/idempotency, invalid metadata, conflicting versions, ambiguous dates, future dates, stale version/hash rejection, unsupported questions, citation validity, groundedness, request linkage, embedding cache reuse, and controlled failure paths.

**Assessment types (recorded on every row):**

* `deterministic` — assertions over data and logs; a failure records FAIL and never aborts the run.
* `model_based` — judged by a real constrained model call; an advisory signal, not ground truth.
* `manual_review` — human review pending; recorded as NOT_EXECUTED and NEVER counted as a pass.

**Statuses:** PASS / FAIL / NOT_EXECUTED. `pass_rate_executed` in `evaluation_run_summary` counts PASS/FAIL only — unsupported or unexecuted assessments can never inflate it.

**Simulated tests are labeled** (`simulated=true`): the embedding-failure fallback uses a monkeypatched embedding call (chat stays real), and the observability-isolation check injects malformed log rows. No exact answer-string assertions are used where model wording may vary: intervals are extracted by pattern, and abstention/groundedness are judged.

**Fixtures:** one ambiguous-date fixture file is written and removed in a try/finally; no other files are generated. Real model calls are limited to: the supported-question answer, its groundedness judge, the unsupported-question answer and judge, and the simulated-fallback answer.

**Outputs:** rows appended to `evaluation_results_raw` (created with its curated views by `10_document_framework`): `evaluation_results` (detail grain: run_id + test_id) and `evaluation_run_summary` (grain: run_id).

In [0]:
%run ./30_retrieval_answers

In [0]:
# --- Evaluation harness -----------------------------------------------------------
# One run_id per execution; every result row is persisted (append-only) with
# run_id, test_id, category, assessment_type, expected/actual outcome, status,
# diagnostics, simulated flag, and timestamp. A test that RAISES records FAIL
# with the error — the run continues.
RUN_ID = str(uuid.uuid4())
RESULTS = []
EVAL_STATE = {}  # cross-test context (e.g., a real request's id for linkage checks)

# Storage (evaluation_results_raw) and the curated views evaluation_results /
# evaluation_run_summary are created by 10_document_framework.ensure_storage()
# (already run by the %run chain above) — nothing to create here.

def record_result(test_id, category, assessment_type, expected, actual, status,
                  diagnostics='', simulated=False):
    RESULTS.append(dict(run_id=RUN_ID, test_id=test_id, category=category,
                        assessment_type=assessment_type, expected_outcome=str(expected),
                        actual_outcome=str(actual), status=status,
                        diagnostics='' if diagnostics is None else str(diagnostics),
                        simulated=simulated, evaluated_at=datetime.now(timezone.utc).isoformat()))

def evaluate(test_id, category, assessment_type, expected, actual, ok,
             diagnostics='', simulated=False):
    record_result(test_id, category, assessment_type, expected, actual,
                  'PASS' if ok else 'FAIL', diagnostics, simulated)

def run_test(test_id, category, assessment_type, expected, fn, simulated=False):
    """Run one check; fn returns (ok, actual_outcome, diagnostics?).

    A raised exception records FAIL with the error instead of aborting the run.
    """
    try:
        ok, actual, *rest = fn()
        evaluate(test_id, category, assessment_type, expected, actual, ok,
                 rest[0] if rest else '', simulated)
    except Exception as exc:
        evaluate(test_id, category, assessment_type, expected,
                 f'error: {type(exc).__name__}: {str(exc)[:200]}', False,
                 'check raised', simulated)

def model_judge(instruction, payload):
    """One constrained real chat call used ONLY for model-based assessment.

    Returns the model's short verdict (lowercased). Model-based assessments are
    advisory signals recorded separately from deterministic assertions — they
    are not ground truth.
    """
    prompt = (instruction + '\nReply with exactly one word and nothing else.\n'
              + json.dumps(payload))
    spark.createDataFrame([(prompt,)], 'prompt string').createOrReplaceTempView('agri_eval_judge')
    out = spark.sql(f"SELECT ai_query('{CHAT_MODEL}', prompt) AS verdict FROM agri_eval_judge").first()['verdict']
    return str(out).strip().lower()

# Curated evaluation views are defined by 10_document_framework (single source
# of truth; see ensure_storage) — this cell only defines the harness functions.
print('Evaluation harness ready. run_id =', RUN_ID)

In [0]:
# --- Deterministic checks: version selection and ingestion rules (no model calls) ---

def _t_latest_active():
    reg = registry_snapshot()
    inv = spark.table(table('document_inventory'))
    active_rows = inv.filter(F.col('version_status')=='ACTIVE').count()
    distinct_docs = inv.select('document_id').distinct().count()
    ok = reg.get('AGR-001', (None,))[0] == 'AGR-001-V3' and active_rows == distinct_docs
    return ok, f"registry AGR-001={reg.get('AGR-001')}, active_rows={active_rows}, distinct_docs={distinct_docs}"
run_test('version_selection.latest_version_active', 'version_selection',
         'AGR-001 active version is AGR-001-V3; exactly one ACTIVE version per document', _t_latest_active)

def _t_changed_filename():
    rows = {r.version_id: r.filename for r in spark.table(table('document_inventory'))
            .filter(F.col('document_id')=='AGR-001').collect()}
    ok = rows.get('AGR-001-V2') == 'revised_guidance_1.md' and rows.get('AGR-001-V3') == 'new_field_manual_1.md'
    return ok, f"filenames={rows}"
run_test('version_selection.changed_filename', 'version_selection',
         'V3 was published from a different filename than V2 under the same document_id', _t_changed_filename)

def _t_late_arrival():
    v1 = spark.table(table('document_manifest')).filter(
        (F.col('document_id')=='AGR-001') & (F.col('version_id')=='AGR-001-V1')).first()
    non_v3 = spark.table(table('active_document_chunks')).filter(
        (F.col('document_id')=='AGR-001') & (F.col('version_id')!='AGR-001-V3')).count()
    ok = v1 is not None and bool(v1.is_active == False) and non_v3 == 0
    return ok, f"V1 active={v1.is_active if v1 else None}, non-V3 active chunks={non_v3}"
run_test('version_selection.late_arrival_inactive', 'version_selection',
         'A late-arriving V1 file is kept as history but never reactivated; no retired chunks in the active pool', _t_late_arrival)

def _t_future_dated():
    row = spark.table(table('document_inventory')).filter(F.col('document_id')=='AGR-006').first()
    ok = row is not None and bool(row.is_active == False) and row.version_status == 'FUTURE_DATED'
    return ok, f"AGR-006 status={getattr(row, 'version_status', None)}"
run_test('version_selection.future_dated_inactive', 'version_selection',
         'AGR-006 is stored as history with FUTURE_DATED status in the inventory view (distinct from retired history)', _t_future_dated)

def _t_idempotency():
    before = (spark.table(table('document_versions')).count(), spark.table(table('document_chunks')).count())
    res1, res2 = ingest_and_publish(), ingest_and_publish()
    after = (spark.table(table('document_versions')).count(), spark.table(table('document_chunks')).count())
    ok = before == after and res1['quarantined_files'] == 0 and res2['quarantined_files'] == 0
    return ok, f"versions/chunks {before}=={after}, quarantined this pass: {res1['quarantined_files']}/{res2['quarantined_files']}"
run_test('ingestion.duplicate_idempotency', 'ingestion',
         'Re-ingesting unchanged files adds no rows and quarantines nothing (duplicates match stored hashes)', _t_idempotency)

def _t_invalid_metadata():
    obs = spark.table(table('quarantine_observations')).filter(
        F.col('source_path').endswith('broken_metadata.md')).collect()
    accepted = spark.table(table('document_versions')).filter(F.col('filename')=='broken_metadata.md').count()
    ok = any('Missing or invalid' in (o.reason or '') for o in obs) and accepted == 0
    return ok, f"observations={len(obs)}, reason={(obs[0].reason if obs else None)}, accepted rows={accepted}"
run_test('ingestion.invalid_metadata_quarantined', 'ingestion',
         'A file with missing required metadata is quarantined with a reason and never accepted', _t_invalid_metadata)

def _t_conflicting():
    obs = {r.source_path.split('/')[-1]: r.reason
           for r in spark.table(table('quarantine_observations')).collect()}
    ok = (obs.get('conflicting_v3.md') == 'Conflicting version content'
         and obs.get('new_field_manual_1.md') == 'Conflicting version content'
         and registry_snapshot()['AGR-001'][0] == 'AGR-001-V3')
    return ok, f"reasons={ {k: v for k, v in obs.items() if k in ('conflicting_v3.md', 'new_field_manual_1.md')} }"
run_test('ingestion.conflicting_version_quarantined', 'ingestion',
         'Conflicting content for an existing (document_id, version_id) quarantines the batch; accepted V3 history stays active', _t_conflicting)

def _t_ambiguous_date():
    fixture = make_document(1, 3).replace('AGR-001-V3', 'AGR-001-EVAL-AMB')  # same date as active V3, new version key
    path = LANDING / 'ambig_date_eval.md'
    try:
        path.write_text(fixture)
        res = ingest_and_publish()
        obs = spark.table(table('quarantine_observations')).filter(
            F.col('source_path').endswith('ambig_date_eval.md')).collect()
        ok = (res['quarantined_files'] >= 1
              and any('Ambiguous document date' in (o.reason or '') for o in obs)
              and registry_snapshot()['AGR-001'][0] == 'AGR-001-V3')
        return ok, f"quarantined this pass={res['quarantined_files']}, reason={(obs[0].reason if obs else None)}, registry unchanged"
    finally:
        if path.exists():
            path.unlink()  # evaluation fixture cleanup ALWAYS runs, even if the check fails
run_test('ingestion.ambiguous_date_quarantined', 'ingestion',
         'A second version claiming the active version business date is quarantined as an ambiguous batch; the active version is untouched', _t_ambiguous_date)

In [0]:
# --- Model-based checks (real model calls; deterministic assertions kept separate) ---

def _t_citation_validity():
    r = ask('What is the irrigation review interval for wheat?', 'AGR-001', context='evaluation')
    EVAL_STATE['supported_request_id'] = r['request_id']
    EVAL_STATE['supported_answer'] = r
    reg = registry_snapshot()
    sources_ok = bool(r['sources']) and all(
        s['document_id'] == 'AGR-001' and s['version_id'] == 'AGR-001-V3'
        and reg.get('AGR-001', (None,))[0] == 'AGR-001-V3'
        for s in r['sources'])
    chunks_ok = all(spark.table(table('document_chunks')).filter(
        (F.col('document_id') == s['document_id']) & (F.col('version_id') == s['version_id'])
        & (F.col('filename') == s['filename'])).count() >= 1 for s in r['sources'])
    m = re.search(r'(\d+)\s*-?\s*days?\b', r['answer'])  # wording-independent interval extraction
    interval_ok = bool(m) and m.group(1) == '6' or bool(re.search(r'\bsix\b', r['answer'].lower()))
    ok = r['mode'] == 'GROUNDED_LLM' and sources_ok and chunks_ok and interval_ok
    return ok, (f"mode={r['mode']}, cited versions={sorted({s['version_id'] for s in r['sources']})}, "
                f"interval={(m.group(1) + ' days' if m else 'six (words)')}"), \
           'assertions are deterministic; evaluated on a real model answer'
run_test('answers.citation_validity', 'answers',
         'The grounded answer cites only AGR-001-V3 with existing files/chunks and states the 6-day interval (pattern-extracted, not string-matched)',
         _t_citation_validity)

def _t_groundedness():
    r = EVAL_STATE['supported_answer']
    verdict = model_judge(
        'Does the ANSWER state only information supported by the EXCERPTS? Answer SUPPORTED or NOT_SUPPORTED.',
        {'answer': r['answer'], 'excerpts': r['evidence']})
    v = verdict.strip('._')
    ok = v == 'supported' or v == 'yes'
    return ok, f"judge verdict={verdict[:40]}", 'model-based assessment; advisory signal, not ground truth'
run_test('answers.groundedness', 'answers',
         'The supported-question answer states only information present in its cited evidence (model-judged)', _t_groundedness)

def _t_unsupported():
    q = 'What is the capital of France?'
    r = ask(q, 'AGR-001', context='evaluation')
    verdict = model_judge(
        ('The ANSWER responds to a question the EXCERPTS do not cover. Does the ANSWER state that the current documents '
         'do not provide the requested information, without inventing an answer? Answer YES or NO.'),
        {'question': q, 'answer': r['answer'], 'excerpts': r['evidence']})
    ok = verdict.strip('._') in ('yes', 'y') and r['mode'] in ('GROUNDED_LLM', 'EXTRACTIVE')
    return ok, f"mode={r['mode']}, judge={verdict[:40]}", 'model-based assessment; advisory signal, not ground truth'
run_test('answers.unsupported_question', 'answers',
         'An unsupported question gets an abstaining answer that invents nothing (model-judged, not string-matched)', _t_unsupported)

In [0]:
# --- Deterministic checks: request linkage and embedding observability (no model calls) ---

def _t_request_linkage():
    rid = EVAL_STATE['supported_request_id']
    req = spark.table(table('request_log')).filter(F.col('request_id') == rid).collect()
    aud = spark.table(table('retrieval_audit')).filter(F.col('request_id') == rid).collect()
    emb = spark.table(table('embedding_log')).filter(F.col('request_id') == rid).collect()
    audit_versions = {a['version_id'] for a in json.loads(aud[0].accepted_json)} if aud else set()
    request_versions = set(json.loads(req[0].accepted_versions_json)) if req else set()
    kinds = {e.embedding_kind for e in emb}
    ok = (len(req) == 1 and len(aud) == 1 and len(emb) >= 1
          and req[0].answer_mode == 'GROUNDED_LLM' and req[0].retrieval_mode == 'SEMANTIC'
          and audit_versions == request_versions == {'AGR-001-V3'}
          and kinds == {'query', 'document'})
    return ok, (f"request_log rows={len(req)}, audit rows={len(aud)}, embedding rows={len(emb)} "
               f"(kinds={sorted(kinds)}), versions={sorted(request_versions)}")
run_test('logging.request_linkage', 'logging',
         'One request produces exactly one request_log row, one retrieval_audit row, and linked embedding_log rows sharing the request_id, with matching accepted versions', _t_request_linkage)

def _t_cache_reuse():
    rows = spark.table(table('embedding_log')).collect()
    submitted = [r for r in rows if r.embedding_kind == 'document' and r.text_submitted]
    reused = [r for r in rows if r.embedding_kind == 'document' and r.cache_reused]
    queries = [r for r in rows if r.embedding_kind == 'query']
    ok = len(submitted) >= 3 and len(reused) >= 3 and len(queries) >= 1
    return ok, (f"document submitted={len(submitted)}, document reused={len(reused)}, query rows={len(queries)} "
                '(application-submitted counts; not provider inference counts)')
run_test('logging.embedding_cache_reuse', 'logging',
         'embedding_log distinguishes document vs query texts and records both real submissions and cache reuse', _t_cache_reuse)

def _t_stale_rejection():
    stale = next(r.asDict() for r in spark.table(table('document_chunks')).filter(
        (F.col('document_id') == 'AGR-001') & (F.col('version_id') == 'AGR-001-V2')).collect())
    kept = validate_candidates([stale], registry_snapshot())
    non_v3 = spark.table(table('active_document_chunks')).filter(
        (F.col('document_id') == 'AGR-001') & (F.col('version_id') != 'AGR-001-V3')).count()
    ok = kept == [] and non_v3 == 0
    return ok, f"stale V2 candidate kept={len(kept)}, non-V3 active chunks={non_v3}", \
           'real validate_candidates call on a real retired chunk (controlled injection; no data writes)'
run_test('retrieval.stale_candidate_rejection', 'retrieval',
         'A retired-version chunk fails registry validation (version+hash) and is never used as evidence', _t_stale_rejection)

In [0]:
# --- Simulated failure paths (controlled injection; labeled simulated=true) ----------

def _t_embedding_failure():
    saved = (native_embedding, embedding_available, _last_embedding_error)
    def _simulate(text):
        raise RuntimeError('SIMULATED embedding service failure (controlled injection)')
    globals()['native_embedding'] = _simulate
    globals()['embedding_available'] = True
    try:
        r = ask('Which operator records the demonstration review date for wheat?', 'AGR-001', context='evaluation')
        row = spark.table(table('request_log')).filter(F.col('request_id') == r['request_id']).first()
        ok = (row is not None and row.retrieval_mode == 'TFIDF'
              and bool(row.error) and 'SIMULATED' in row.error and bool(row.fallback_reason)
              and r['mode'] == 'GROUNDED_LLM'
              and json.loads(row.accepted_versions_json) == ['AGR-001-V3']
              and all(s['version_id'] == 'AGR-001-V3' for s in r['sources'])
              and registry_snapshot()['AGR-001'][0] == 'AGR-001-V3')
        return ok, (f"retrieval={row.retrieval_mode if row else None}, answer={row.answer_mode if row else None}, "
                    f"error={(row.error[:60] if row and row.error else None)}, fallback={(row.fallback_reason if row else None)}"), \
               'controlled injection: native_embedding monkeypatched to fail; chat model still real'
    finally:
        globals()['native_embedding'], globals()['embedding_available'], globals()['_last_embedding_error'] = saved
run_test('resilience.embedding_failure_fallback', 'resilience',
         'An injected embedding failure degrades to the labeled TF-IDF fallback, is captured in request_log with an error and fallback reason, and never changes version eligibility or citations',
         _t_embedding_failure, simulated=True)

def _t_observability_isolation():
    before = spark.table(table('request_log')).count()
    try:
        _log_request_row({'request_id': 'isolation-probe', 'rejected_stale_candidates': 'not-a-number'})  # malformed on purpose
        _log_embedding_rows('isolation-probe', 'evaluation', [{'kind': 'query'}])  # malformed on purpose
        raised = False
    except Exception:
        raised = True  # must never happen: logging is best-effort and never raises
    leaked = spark.table(table('request_log')).filter(F.col('request_id') == 'isolation-probe').count()
    ok = (not raised) and leaked == 0 and registry_snapshot()['AGR-001'][0] == 'AGR-001-V3'
    return ok, f"raised={raised}, leaked rows={leaked}, registry unchanged", \
           'controlled injection: malformed log rows; version eligibility unaffected'
run_test('logging.observability_isolation', 'logging',
         'Malformed log rows are skipped with a warning — observability failures never raise and never alter version eligibility',
         _t_observability_isolation, simulated=True)

In [0]:
# --- Manual-review entries (never counted as passes) and run persistence -------------
record_result('manual.answer_style_review', 'manual_review', 'manual_review',
              'Human review of answer phrasing quality across questions',
              'review pending', 'NOT_EXECUTED',
              'manual assessment by design; excluded from pass_rate_executed')
record_result('manual.source_material_accuracy', 'manual_review', 'manual_review',
              'Domain-expert review of the fictional agronomy content',
              'review pending', 'NOT_EXECUTED',
              'manual assessment by design; excluded from pass_rate_executed')

spark.createDataFrame(RESULTS, EVAL_RESULTS_SCHEMA).write.mode('append').saveAsTable(table('evaluation_results_raw'))
print(f"Persisted {len(RESULTS)} evaluation results for run {RUN_ID}")

display(spark.table(table('evaluation_results')).filter(F.col('run_id') == RUN_ID)
        .select('test_id', 'category', 'assessment_type', 'status', 'simulated'))

summary = spark.table(table('evaluation_run_summary')).filter(F.col('run_id') == RUN_ID).first()
display(spark.table(table('evaluation_run_summary')).orderBy(F.desc('run_started')))

assert summary is not None
assert summary.failed == 0, 'An executed evaluation test failed — see evaluation_results for this run'
assert summary.passed == summary.deterministic_tests + summary.model_based_tests, 'An executed test did not pass'
assert summary.not_executed == summary.manual_review_tests == 2, 'Manual reviews must be marked NOT_EXECUTED'
assert summary.simulated_tests == 2, 'Simulated tests must be labeled'
assert summary.pass_rate_executed == 1.0, 'Executed pass rate must be computed from PASS/FAIL only'
assert spark.table(table('evaluation_results')).filter(
    (F.col('run_id') == RUN_ID) & F.col('simulated') & (F.col('status') != 'PASS')).count() == 0
print('PASS: evaluation run complete — all executed assessments passed; simulated tests labeled; '
      'manual reviews marked NOT_EXECUTED (never counted as passes).')